# Treinamento dos modelos
* Os modelos serão treinados com base na métrica de 'recall', para identificar posteriormente quais variáveis foram mais determinantes para a identificação de uma vitória ou derrota.

In [1]:
# Biblioteca de manipulação de dados
import pandas as pd
import numpy as np
from scipy.stats import loguniform, uniform, randint

# Preparação dos dados
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import RandomizedSearchCV, cross_val_score

# Biblioteca de modelos
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from xgboost import XGBClassifier
import keras_tuner as kt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from scikeras.wrappers import KerasClassifier
from tensorflow.keras import Input

# Bibliotecas de avaliação do modelo
from sklearn.metrics import recall_score, roc_auc_score, roc_curve, classification_report, confusion_matrix
import plotly.express as px
import shap

In [2]:
df = pd.read_csv('../src/data/cleaned/LOL_limpo.csv')
df.head()

,blueWins,blueWardsPlaced,blueWardsDestroyed,blueFirstBlood,blueKills,blueDeaths,blueAssists,blueEliteMonsters,blueDragons,blueHeralds,...,redTowersDestroyed,redTotalGold,redAvgLevel,redTotalExperience,redTotalMinionsKilled,redTotalJungleMinionsKilled,redGoldDiff,redExperienceDiff,redCSPerMin,redGoldPerMin
0,0,28,2,1,9,6,11,0,0,0,...,0,16567,6.8,17047,197,55,-643,8,19.7,1656.7
1,0,15,0,0,7,11,4,1,1,0,...,0,17285,6.8,17254,203,28,1172,1033,20.3,1728.5
2,0,43,1,0,4,5,5,1,0,1,...,0,16478,7.0,17961,235,47,1321,7,23.5,1647.8
3,0,75,4,0,6,6,6,0,0,0,...,0,17404,7.0,18313,225,67,1004,-230,22.5,1740.4
4,1,18,0,0,5,3,6,1,1,0,...,0,15201,7.0,18060,221,59,-698,-101,22.1,1520.1


In [3]:
df.describe()

,blueWins,blueWardsPlaced,blueWardsDestroyed,blueFirstBlood,blueKills,blueDeaths,blueAssists,blueEliteMonsters,blueDragons,blueHeralds,...,redTowersDestroyed,redTotalGold,redAvgLevel,redTotalExperience,redTotalMinionsKilled,redTotalJungleMinionsKilled,redGoldDiff,redExperienceDiff,redCSPerMin,redGoldPerMin
count,7924.000000,7924.000000,7924.000000,7924.000000,7924.000000,7924.000000,7924.000000,7924.000000,7924.000000,7924.000000,...,7924.0,7924.000000,7924.000000,7924.000000,7924.000000,7924.000000,7924.000000,7924.000000,7924.000000,7924.000000
mean,0.501010,19.699016,2.697501,0.502524,6.012998,5.970217,6.436648,0.532181,0.361055,0.171126,...,0.0,16384.373423,6.932408,17986.491418,218.272211,51.477915,-1.742428,40.449142,21.827221,1638.437342
std,0.500031,10.235367,1.689964,0.500025,2.789191,2.733365,3.728194,0.616652,0.480337,0.376643,...,0.0,1325.169249,0.278480,1094.993966,20.774028,9.709273,2115.213407,1720.530448,2.077403,132.516925
min,0.000000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.0,12626.000000,6.200000,14418.000000,153.000000,22.000000,-6547.000000,-5575.000000,15.300000,1262.600000
25%,0.000000,14.000000,1.000000,0.000000,4.000000,4.000000,4.000000,0.000000,0.000000,0.000000,...,0.0,15418.750000,6.800000,17257.500000,204.000000,44.000000,-1446.750000,-1143.000000,20.400000,1541.875000
50%,1.000000,16.000000,3.000000,1.000000,6.000000,6.000000,6.000000,0.000000,0.000000,0.000000,...,0.0,16313.000000,7.000000,17973.000000,219.000000,52.000000,-1.500000,36.000000,21.900000,1631.300000
75%,1.000000,19.000000,4.000000,1.000000,8.000000,8.000000,9.000000,1.000000,1.000000,0.000000,...,0.0,17275.750000,7.200000,18737.250000,233.000000,58.000000,1449.000000,1209.250000,23.300000,1727.575000
max,1.000000,76.000000,9.000000,1.000000,15.000000,14.000000,18.000000,2.000000,1.000000,1.000000,...,0.0,20920.000000,7.800000,21517.000000,282.000000,81.000000,6717.000000,5585.000000,28.200000,2092.000000


In [4]:
# Tirando colunas zeradas
df = df.drop(columns=['blueTowersDestroyed'])
df = df.drop(columns=['redTowersDestroyed'])

In [5]:
# Como o describe revelou que a distribuição está bem parecida da coluna target, não vou balancear
x = df.drop(columns=['blueWins'])
y = df['blueWins'].values

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42)

In [6]:
# Ajuste de escala dos dados
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)

In [7]:
# Ajuste dos testes
x_test_scaled = scaler.transform(x_test)

In [8]:
# Tuning de hiperparâmetros do modelo neural
def build_rl(hp):
    rl = Sequential()
    rl.add(Dense(
        units = hp.Int("units1", min_value=32, max_value=128, step=32),
        activation="relu",
        input_shape=(x_train_scaled.shape[1],)
    ))
    rl.add(Dense(
        units=hp.Int("units2", min_value=32, max_value=128, step=32),
        activation="relu",
    ))
    rl.add(Dense(1, activation="sigmoid"))
    rl.compile(
        optimizer=Adam(
            learning_rate=hp.Choice('learning_rate', [0.001, 0.01, 0.1])
        ),
        loss="binary_crossentropy",
        metrics=[tf.keras.metrics.Recall()]
    )
    return rl
tuner = kt.RandomSearch(
    build_rl,
    objective='val_recall',
    max_trials=10,
    executions_per_trial=2,
    project_name='Modelagem_rl',
    directory='rl_tuner'
)
tuner.search(x_train_scaled, y_train, epochs=50, validation_split=0.2)
best_rl = tuner.get_best_hyperparameters(num_trials=1)[0]
best_rl.values

Reloading Tuner from rl_tuner\Modelagem_rl\tuner0.json


{'units1': 32, 'units2': 128, 'learning_rate': 0.1}

In [22]:
# Modelo de regressão logística com cross validation
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = []
for train_idx, val_idx in kf.split(x_train_scaled, y_train):
    x_tr, x_val = x_train_scaled[train_idx], x_train_scaled[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]

    # Recria o modelo com os hiperparâmetros encontrados
    rl = Sequential([
        Input(shape=(x_tr.shape[1],)),
        Dense(128, activation='relu'),
        Dense(1, activation='sigmoid')
    ])

    rl.compile(
        optimizer=Adam(learning_rate=0.1),
        loss='binary_crossentropy',
        metrics=[tf.keras.metrics.Recall()]
    )

    rl.fit(x_tr, y_tr, batch_size=32, epochs=300, verbose=0)
    scores.append(rl.evaluate(x_val, y_val, verbose=0)[1])  # recall

print("Regressão Logística (recall):", np.mean(scores).round(2))

Regressão Logística (recall): 0.68


In [10]:
# Treino do modelo de Random Forest
param_rf = {
    'n_estimators': randint(50, 200),
    'max_depth': [3, 10, 20, 30, 40, 50],
    'min_samples_split': [2, 4, 8, 10, 12, 16, 18, 20],
    'min_samples_leaf': [1,2,4,5,7,8,10],
    'criterion': ['gini', 'entropy'],
    'max_features': ['sqrt', 'log2'],
}
search_rf = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_rf,
    n_iter=15,
    scoring='recall',
    cv=5,
    n_jobs=-1,
    random_state=42,
    refit=True
)
search_rf.fit(x_train_scaled, y_train)
best_rf = search_rf.best_estimator_
scores_rf = cross_val_score(
    best_rf,
    x_train_scaled,
    y_train,
    cv=5,
    n_jobs=-1,
    scoring='recall'
)
print('Random Forest (recall):', scores_rf.mean().round(2))

Random Forest (recall): 0.72


In [11]:
# Treino do modelo de XGBoost

param_xgb = {
    'n_estimators': randint(100, 1000),
    'max_depth': randint(3, 15),
    'learning_rate': uniform(0.01, 0.29),
    'subsample': uniform(0.5, 0.5),
    'colsample_bytree': uniform(0.5, 0.5),
    'gamma': uniform(0, 10),
    'reg_alpha': uniform(0, 10),
    'reg_lambda': uniform(0, 10)
}
search_xgb = RandomizedSearchCV(
    estimator=XGBClassifier(random_state=42),
    param_distributions=param_xgb,
    n_iter=15,
    scoring='recall',
    cv=5,
    n_jobs=-1,
    random_state=42,
    refit=True
)
search_xgb.fit(x_train_scaled, y_train)
best_xgb = search_xgb.best_estimator_
scores_xgb = cross_val_score(
    best_xgb,
    x_train_scaled,
    y_train,
    cv=5,
    n_jobs=-1,
    scoring='recall'
)
print('XGBoost (recall):', scores_xgb.mean().round(2))

XGBoost (recall): 0.72


In [23]:
# Teste do modelo de Rl neural
y_pred_rl = (rl.predict(x_test_scaled) > 0.5).astype(int)

# Avaliação
report_rl = classification_report(y_test, y_pred_rl)
print("Relatório de classificação de Rede neural")
print(report_rl)

# Matriz de confusão
conf_rl = confusion_matrix(y_test, y_pred_rl)
classes = ['Perdeu', 'Venceu']

fig = px.imshow(
    conf_rl,
    text_auto=True,
    aspect='auto',
    template='simple_white',
    x=classes,
    y=classes,
    color_continuous_scale='Blues',
    labels=dict(x='Predição', y='Real', color='Contagem'),
    title= 'Matriz de confusão - Regressão Logística'
)
fig.show()

75/75 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Relatório de classificação de Regressão logística
              precision    recall  f1-score   support

           0       0.71      0.76      0.73      1214
           1       0.73      0.68      0.70      1164

    accuracy                           0.72      2378
   macro avg       0.72      0.72      0.72      2378
weighted avg       0.72      0.72      0.72      2378



In [13]:
# Teste do modelo de Random Forest
y_pred_rf = best_rf.predict(x_test_scaled)

# Avaliação do modelo com o teste
report_rf = classification_report(y_test, y_pred_rf)
print('Relatório de classificação do Random Forest')
print(report_rf)

# Matriz de confusão
conf_rf = confusion_matrix(y_test, y_pred_rf)
classes = ['Perdeu', 'Venceu']

fig = px.imshow(
    conf_rf,
    text_auto=True,
    aspect='auto',
    template='simple_white',
    x=classes,
    y=classes,
    labels= dict(x='Predição', y='Real', color='Contagem'),
    color_continuous_scale= 'Blues',
    title='Matriz de confusão - Random Forest'

)
fig.show()

Relatório de classificação do Random Forest
              precision    recall  f1-score   support

           0       0.74      0.68      0.71      1214
           1       0.69      0.74      0.72      1164

    accuracy                           0.71      2378
   macro avg       0.71      0.71      0.71      2378
weighted avg       0.71      0.71      0.71      2378



In [14]:
# Teste do modelo de XGBoost
y_pred_xgb = best_xgb.predict(x_test_scaled)

# Avaliação do modelo com o teste
report_xgb = classification_report(y_test, y_pred_xgb)
print('Relatório de classificação XGBoost')
print(report_xgb)

# Matriz de confusão
conf_xgb = confusion_matrix(y_test, y_pred_xgb)
classes = ['Perdeu', 'Venceu']

fig = px.imshow(
    conf_xgb,
    text_auto=True,
    aspect='auto',
    template='seaborn',
    x=classes,
    y=classes,
    labels= dict(x='Predição', y='Real', color='Contagem'),
    color_continuous_scale= 'Blues',
    title='Matriz de confusão - XGBoost'

)
fig.show()

Relatório de classificação XGBoost
              precision    recall  f1-score   support

           0       0.73      0.69      0.71      1214
           1       0.69      0.74      0.72      1164

    accuracy                           0.71      2378
   macro avg       0.71      0.71      0.71      2378
weighted avg       0.71      0.71      0.71      2378



In [19]:
def build_rl_stack():
    rl_stack = Sequential([
    Input(shape=(x_train_scaled.shape[1],)),
    Dense(32, activation='relu'),
    Dense(128, activation='relu'),
    Dense(1, activation='sigmoid')
    ])
    rl_stack.compile(optimizer=Adam(learning_rate=0.1),
               loss='binary_crossentropy',
               metrics=["Recall"])
    return rl_stack

# Adaptando o modelo neural para scikit
nn_model = KerasClassifier(model=build_rl_stack, epochs=400, batch_size=32, verbose=0)

# Stacking dos modelos
stack = StackingClassifier(
    estimators=[('nn', nn_model)],
    final_estimator=best_xgb
)

stack.fit(x_train_scaled, y_train)
print("Recall Stacking: ", round(stack.score(x_test_scaled, y_test), 2))

Recall Stacking:  0.67


In [35]:
# Usando Shap para entender importância das features
explainer = shap.TreeExplainer(best_xgb, feature_perturbation="tree_path_dependent")
shap_values = explainer(x_test_scaled)

importances = np.abs(shap_values.values).mean(axis=0)
features = x.columns

df_shap = pd.DataFrame({
    "Feature": features,
    "Importances": importances
}).sort_values("Importances", ascending=False).head(10)

fig = px.bar(
    df_shap,
    x="Importances",
    y="Feature",
    orientation='h',
    labels={'x': 'Feature', 'y': 'Importância média'},
    title='Importâcia das variáveis'
)
fig.update_layout(yaxis={'categoryorder':'total ascending'})
fig.show()

In [37]:
# Testando o modelo com as 10 variáveis selecionadas pelo Shap
lista = ['blueGoldDiff', 'blueDragons', 'blueExperienceDiff', 'redGoldDiff', 'redDragons', 'redExperienceDiff', 'blueAvgLevel', 'blueEliteMonsters', 'redTotalGold', 'blueTotalGold']

x_shap = df[lista]

x_train_shap, x_test_shap, y_train_shap, y_test_shap = train_test_split(x_shap, y, test_size=0.3, random_state=42)

In [38]:
# Treino do modelo de XGBoost com Shap

param_xgbs = {
    'n_estimators': randint(100, 1000),
    'max_depth': randint(3, 15),
    'learning_rate': uniform(0.01, 0.29),
    'subsample': uniform(0.5, 0.5),
    'colsample_bytree': uniform(0.5, 0.5),
    'gamma': uniform(0, 10),
    'reg_alpha': uniform(0, 10),
    'reg_lambda': uniform(0, 10)
}
search_xgbs = RandomizedSearchCV(
    estimator=XGBClassifier(random_state=42),
    param_distributions=param_xgbs,
    n_iter=15,
    scoring='recall',
    cv=5,
    n_jobs=-1,
    random_state=42,
    refit=True
)
search_xgbs.fit(x_train_shap, y_train_shap)
best_xgbs = search_xgbs.best_estimator_
scores_xgbs = cross_val_score(
    best_xgbs,
    x_train_shap,
    y_train_shap,
    cv=5,
    n_jobs=-1,
    scoring='recall'
)
print('XGBoost (recall):', scores_xgb.mean().round(2))

XGBoost (recall): 0.72


In [39]:
# Teste do modelo de XGBoost com Shap
y_pred_xgbs = best_xgbs.predict(x_test_shap)

# Avaliação do modelo com o teste
report_xgbs = classification_report(y_test_shap, y_pred_xgbs)
print('Relatório de classificação XGBoost')
print(report_xgbs)

# Matriz de confusão
conf_xgbs = confusion_matrix(y_test_shap, y_pred_xgbs)
classes = ['Perdeu', 'Venceu']

fig = px.imshow(
    conf_xgbs,
    text_auto=True,
    aspect='auto',
    template='seaborn',
    x=classes,
    y=classes,
    labels= dict(x='Predição', y='Real', color='Contagem'),
    color_continuous_scale= 'Blues',
    title='Matriz de confusão - XGBoost'

)
fig.show()

Relatório de classificação XGBoost
              precision    recall  f1-score   support

           0       0.73      0.69      0.71      1214
           1       0.69      0.74      0.71      1164

    accuracy                           0.71      2378
   macro avg       0.71      0.71      0.71      2378
weighted avg       0.71      0.71      0.71      2378



In [40]:
# Parâmetros utilizados para o modelo
print(search_xgbs.best_params_)

{'colsample_bytree': np.float64(0.6872700594236812), 'gamma': np.float64(9.50714306409916), 'learning_rate': np.float64(0.22227824312530747), 'max_depth': 7, 'n_estimators': 714, 'reg_alpha': np.float64(4.458327528535912), 'reg_lambda': np.float64(0.9997491581800289), 'subsample': np.float64(0.7296244459829335)}


# Resultados
* O modelo a ser usado é o XGBoost, porque obteve o melhor score dentre todos os testados;
* Não fez diferença a seleção limitada de features;

In [2]:
# Testando o modelo com a base original
df_base = pd.read_csv(r"D:\PyCharm 2026.1.4\Projeto_LOL\src\data\raw\Base_LOL.csv")

In [3]:
target = "blueWins"
x_base = df_base.drop(columns=[target])
y_base = df_base[target]

xb_train, xb_test, yb_train, yb_test = train_test_split(x_base, y_base, test_size=0.2, random_state=42)

In [4]:
param_xgbb = {
    'n_estimators': randint(100, 1000),
    'max_depth': randint(3, 15),
    'learning_rate': uniform(0.01, 0.29),
    'subsample': uniform(0.5, 0.5),
    'colsample_bytree': uniform(0.5, 0.5),
    'gamma': uniform(0, 10),
    'reg_alpha': uniform(0, 10),
    'reg_lambda': uniform(0, 10)
}
search_xgbb = RandomizedSearchCV(
    estimator=XGBClassifier(random_state=42),
    param_distributions=param_xgbb,
    n_iter=15,
    scoring='recall',
    cv=5,
    n_jobs=-1,
    random_state=42,
    refit=True
)
search_xgbb.fit(xb_train, yb_train)
best_xgbb = search_xgbb.best_estimator_
scores_xgbb = cross_val_score(
    best_xgbb,
    xb_train,
    yb_train,
    cv=5,
    n_jobs=-1,
    scoring='recall'
)
print('XGBoost (recall):', scores_xgbb.mean().round(2))

XGBoost (recall): 0.71
